# 📖 Notebook 1: Geospatial Search for Nearby Businesses

The most important feature of Yelp is **finding businesses near you**. When a user searches "pizza near me," the system needs to quickly find businesses within a geographic radius.

The problem? Traditional database indexes (B-trees) don't work well for 2D spatial data like latitude/longitude. This notebook explores **why** and shows you **what does work**.

## Learning Objectives

By the end of this notebook, you'll understand:
- Why a naive lat/lon query is slow
- How PostGIS geospatial indexes solve this
- How Elasticsearch geo_distance queries work
- How to cache geospatial search results in Redis
- How to combine location + category filters

## 🛠️ Setup

Start the infrastructure first:

```bash
cd 06-system-designs/yelp
docker compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `yelp_demo`
- **RedisInsight** (Redis GUI): http://localhost:5540  
  Click "Add Redis Database" → Host `redis`, Port `6379`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
import redis
import json
import time
from elasticsearch import Elasticsearch

# -- Connection settings --
DB_CONFIG = {
    "host": "localhost", "port": 5432,
    "database": "yelp_demo", "user": "demo", "password": "demo"
}
REDIS_CONFIG = {"host": "localhost", "port": 6379, "decode_responses": True}
ES_URL = "http://localhost:9200"

def get_db():
    return psycopg2.connect(**DB_CONFIG)

def get_redis():
    return redis.Redis(**REDIS_CONFIG)

def get_es():
    return Elasticsearch(ES_URL)

# Test connections
try:
    conn = get_db(); conn.close()
    print("✅ PostgreSQL connected")
except Exception as e:
    print(f"❌ PostgreSQL: {e}\n   Run: docker compose up -d")

try:
    r = get_redis(); r.ping()
    print("✅ Redis connected")
except Exception as e:
    print(f"❌ Redis: {e}\n   Run: docker compose up -d")

try:
    es = get_es(); es.info()
    print("✅ Elasticsearch connected")
except Exception as e:
    print(f"❌ Elasticsearch: {e}\n   Run: docker compose up -d")

### ⚠️ First: is the data self-consistent?

Every business row stores its position **twice** — once as plain `latitude`/`longitude` columns,
and once as a PostGIS `location` geography. That redundancy is the whole reason this notebook
can compare a naive box against a spatial index, and it is also a trap: the naive query reads
the columns, `ST_DWithin` reads `location`, and Elasticsearch is loaded from the columns. If the
two ever disagree, the three approaches quietly answer *different questions* and every
comparison below is meaningless.

`db/init.sql` now derives `location` from the columns in a single statement so they cannot
drift. But the Postgres volume survives `docker compose down`, so a database seeded by an
older `init.sql` may still be sitting on disk. This cell checks, repairs if needed, and then
asserts — a lab that compares two indexes has no business starting from inconsistent data.

In [ ]:
# Guard: `location` must be the same point as (latitude, longitude).

DRIFT_SQL = """
    SELECT COUNT(*) FROM businesses
    WHERE location IS NULL
       OR ST_Distance(location,
                      ST_SetSRID(ST_MakePoint(longitude, latitude), 4326)::geography) > 1.0
"""

conn = get_db()
cur = conn.cursor()
cur.execute(DRIFT_SQL)
drifted = cur.fetchone()[0]

if drifted:
    print(f"⚠️  {drifted} of the businesses have a `location` that disagrees with their lat/lon.")
    print("   That means this volume was seeded by an older init.sql which drew the geography")
    print("   column from a separate random() call. Repairing in place...")
    cur.execute("""
        UPDATE businesses
        SET location = ST_SetSRID(ST_MakePoint(longitude, latitude), 4326)::geography;
    """)
    conn.commit()
    cur.execute(DRIFT_SQL)
    drifted = cur.fetchone()[0]
    print(f"   ✅ Repaired. Remaining disagreements: {drifted}")
else:
    print("✅ `location` agrees with the latitude/longitude columns for every business.")

assert drifted == 0, (
    f"{drifted} businesses still have a `location` that disagrees with their lat/lon columns — "
    "recreate the database with `docker compose down -v && docker compose up -d`")

conn.close()

## 🤔 The Problem: Finding Nearby Businesses

Imagine a user in **Manhattan** (latitude 40.758, longitude -73.985) searching for restaurants within **2 km**.

The naive approach: check every business and calculate distance.

```sql
-- This is SLOW — full table scan on every query!
SELECT * FROM businesses
WHERE latitude > 40.74 AND latitude < 40.78
  AND longitude > -74.00 AND longitude < -73.97;
```

**Why is this bad?**
- B-tree indexes handle **one dimension** well, but latitude + longitude is **two dimensions**
- The database can use the index for latitude OR longitude, but not both efficiently
- At 10M businesses, this means scanning millions of rows

**And it isn't even the right answer.** A lat/lon rectangle is not a circle, and a degree of
longitude is not a degree of latitude:

- 1° of **latitude** is ~111.3 km everywhere on Earth.
- 1° of **longitude** is ~111.3 km × cos(latitude) — about 84 km in Manhattan, 57 km in
  Stockholm, and 0 km at the poles.

So a box built with the *same* delta on both axes is too narrow east–west (it **misses**
businesses that really are inside the radius) and too generous at the corners (it **returns**
businesses that are not). We reproduce both failures below before fixing them.

Let's see this in action.

In [ ]:
# The NAIVE approach: bounding box with plain columns.
# This is deliberately the *buggy* version — the same delta on both axes. Keep an eye on
# `delta`: we reuse it for longitude even though a degree of longitude is shorter than a
# degree of latitude everywhere except the equator. The next section measures the damage.

conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

# User is in Manhattan
user_lat, user_lon = 40.758, -73.985
# Rough bounding box for a 2 km radius (0.018 degrees of latitude ≈ 2.0 km)
delta = 0.018

start = time.time()
cur.execute("""
    SELECT id, name, city, latitude, longitude, avg_rating, num_reviews
    FROM businesses
    WHERE latitude BETWEEN %s AND %s
      AND longitude BETWEEN %s AND %s
    ORDER BY avg_rating DESC
    LIMIT 10;
""", (user_lat - delta, user_lat + delta, user_lon - delta, user_lon + delta))
naive_results = cur.fetchall()
naive_time = (time.time() - start) * 1000

print(f"📍 Naive bounding box search: {naive_time:.2f} ms")
print(f"   Found {len(naive_results)} businesses near Manhattan\n")
for b in naive_results[:5]:
    print(f"   ⭐ {b['avg_rating']} | {b['name']} ({b['city']})")

# Show the query plan — notice it doesn't use a spatial index
cur.execute("""
    EXPLAIN ANALYZE
    SELECT id, name, city, latitude, longitude
    FROM businesses
    WHERE latitude BETWEEN %s AND %s
      AND longitude BETWEEN %s AND %s;
""", (user_lat - delta, user_lat + delta, user_lon - delta, user_lon + delta))
plan = cur.fetchall()
naive_plan = "\n".join(row["QUERY PLAN"] for row in plan)
print("\n📊 Query Plan (naive):")
for row in plan:
    print(f"   {row['QUERY PLAN']}")

# There is no index on (latitude, longitude), so this MUST be a sequential scan.
# If this assertion ever fires, someone added an index and the lesson here is stale.
assert "Seq Scan" in naive_plan, f"expected a Seq Scan for the un-indexed lat/lon box, got:\n{naive_plan}"

conn.close()

## 🌍 Better: PostGIS Geospatial Index

Our `init.sql` created a **geography** column with a **GIST index** (a spatial index).

PostGIS uses an **R-tree** under the hood — a data structure designed for multi-dimensional data. It groups nearby points into bounding rectangles, making it extremely fast to find points within a radius.

```
B-tree (1D):  ─────────────────────────────────►
              Only good for one dimension at a time

R-tree (2D):  ┌──────┐  ┌────┐
              │ •  • │  │ •  │    Groups nearby points
              │   •  │  │  • │    into rectangles
              └──────┘  └────┘
```

The key function is `ST_DWithin(location, point, distance_in_meters)` — it finds all points within a given radius using the spatial index.

In [ ]:
# The POSTGIS approach: use ST_DWithin with the geography column

conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

user_lat, user_lon = 40.758, -73.985  # Manhattan
radius_meters = 2000  # 2 km

start = time.time()
cur.execute("""
    SELECT
        id, name, city, avg_rating, num_reviews, price_range,
        ST_Distance(location, ST_SetSRID(ST_MakePoint(%s, %s), 4326)::geography) AS distance_m
    FROM businesses
    WHERE ST_DWithin(
        location,
        ST_SetSRID(ST_MakePoint(%s, %s), 4326)::geography,
        %s
    )
    ORDER BY distance_m ASC
    LIMIT 10;
""", (user_lon, user_lat, user_lon, user_lat, radius_meters))
postgis_results = cur.fetchall()
postgis_time = (time.time() - start) * 1000

print(f"📍 PostGIS ST_DWithin search: {postgis_time:.2f} ms")
print(f"   Found {len(postgis_results)} businesses within {radius_meters}m of Manhattan\n")

for b in postgis_results:
    dist_km = b['distance_m'] / 1000
    price = '$' * (b['price_range'] or 1)
    print(f"   📍 {dist_km:.1f} km | ⭐ {b['avg_rating']} ({b['num_reviews']} reviews) | {price} | {b['name']}")

# Show the query plan — now it uses the GIST spatial index!
# Show the query plan. Honest caveat: this table holds 500 rows, so the planner may well
# decide a sequential scan is cheaper than descending the GIST index — at this size it
# genuinely is. The index is what makes the query survive 10M rows, not what makes it fast
# at 500. To show the GIST plan actually exists, we ask the planner to avoid seq scans.
EXPLAIN_SQL = """
    EXPLAIN ANALYZE
    SELECT id, name
    FROM businesses
    WHERE ST_DWithin(
        location,
        ST_SetSRID(ST_MakePoint(%s, %s), 4326)::geography,
        %s
    );
"""
cur.execute(EXPLAIN_SQL, (user_lon, user_lat, radius_meters))
natural_plan = "\n".join(row["QUERY PLAN"] for row in cur.fetchall())
print("\n📊 Query Plan (PostGIS — the planner's own choice at 500 rows):")
for line in natural_plan.split("\n"):
    print(f"   {line}")

cur.execute("SET enable_seqscan = off;")
cur.execute(EXPLAIN_SQL, (user_lon, user_lat, radius_meters))
indexed_plan = "\n".join(row["QUERY PLAN"] for row in cur.fetchall())
cur.execute("SET enable_seqscan = on;")
print("\n📊 Query Plan (PostGIS — seq scan disabled; this is the 10M-row plan):")
for line in indexed_plan.split("\n"):
    print(f"   {line}")

assert "idx_businesses_location" in indexed_plan, (
    f"expected the GIST index idx_businesses_location to be usable, got:\n{indexed_plan}")

print("\n💡 The GIST index gives PostGIS a plan that never reads most of the table.")
print("   At 500 rows the planner may skip it; at 10M rows it is the only survivable plan.")

conn.close()

## 📐 A Box Is Not a Circle: Reproducing Both Failure Modes

The naive query above used `delta = 0.018` on **both** axes. That looks symmetric, but it isn't.
Measured on the WGS-84 spheroid at latitude 40.758:

| Axis | What 0.018° is worth |
|------|----------------------|
| North–south | ≈ **2,000 m** ✅ |
| East–west   | ≈ **1,520 m** ❌ (a degree of longitude here is only 84.4 km, not 111 km) |

So the "2 km box" is only about 1.5 km wide east–west. Every business between ~1,520 m and
2,000 m due east or west of the user is inside the radius and **outside the box** — a false
negative. Meanwhile the box's corners sit ~2,510 m out, so they return businesses that are
**outside** the radius — a false positive.

Both failures are pure geometry, so we can reproduce them deterministically with two probe
points rather than hoping the seed data happens to contain an example. PostGIS measures the
distances, so the arithmetic is checked by the database rather than asserted by us.

**The fix is two-part, and it's the same fix every real system uses:**

1. Size the box so it *contains* the circle. That means dividing by the **smallest** number of
   metres a degree can be worth, not the average: a prefilter that is slightly too big costs a
   few wasted candidates, while one that is slightly too small loses businesses silently.
2. Run an exact distance test on the survivors. That's the "second-pass filter" everyone
   mentions in interviews, and it's what `ST_DWithin` does internally: a cheap bounding-box
   lookup against the index, then an exact distance check.

In [ ]:
import math

# --- Degrees to metres, honestly ---
#
# A degree is not a fixed number of metres, and the variation is exactly what breaks naive
# bounding boxes:
#   * a degree of LATITUDE runs from 110,574 m at the equator to 111,694 m at the poles
#     (the Earth is flattened, so the meridian curves more gently near the poles);
#   * a degree of LONGITUDE is ~111,320 m at the equator and shrinks with cos(latitude).
#
# For a PREFILTER we always divide by the SMALLEST plausible value, plus a small margin. A box
# that is 1% too big costs a handful of extra candidates that the exact pass throws away. A box
# that is 0.1% too small silently loses businesses, and nobody ever files that bug.
M_PER_DEG_LAT_MIN = 110_574.0   # equatorial meridian degree — the shortest one there is
M_PER_DEG_LON_EQ = 111_320.0    # equatorial parallel degree, scaled by cos(latitude) below
SAFETY = 1.01

radius_m = 2000
user_lat, user_lon = 40.758, -73.985
cos_lat = math.cos(math.radians(user_lat))

naive_delta = 0.018                                              # what the cell above used, both axes
lat_delta = SAFETY * radius_m / M_PER_DEG_LAT_MIN                # correct north-south half-height
lon_delta = SAFETY * radius_m / (M_PER_DEG_LON_EQ * cos_lat)     # correct east-west half-width

conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

def distance_to_user(lat, lon):
    """Ask PostGIS for the true spheroid distance from the user to an arbitrary point."""
    cur.execute("""
        SELECT ST_Distance(
            ST_SetSRID(ST_MakePoint(%s, %s), 4326)::geography,
            ST_SetSRID(ST_MakePoint(%s, %s), 4326)::geography
        ) AS d;
    """, (user_lon, user_lat, lon, lat))
    return cur.fetchone()["d"]

# Measure the naive box rather than trusting a constant.
naive_ns_m = distance_to_user(user_lat + naive_delta, user_lon)
naive_ew_m = distance_to_user(user_lat, user_lon + naive_delta)
fixed_ns_m = distance_to_user(user_lat + lat_delta, user_lon)
fixed_ew_m = distance_to_user(user_lat, user_lon + lon_delta)

print(f"📐 Half-extents of each box, measured by PostGIS (target: {radius_m} m)")
print(f"   naive  ({naive_delta:.5f}° both axes) : north-south {naive_ns_m:7.0f} m | east-west {naive_ew_m:7.0f} m  ← {radius_m - naive_ew_m:.0f} m short")
print(f"   fixed  ({lat_delta:.5f}° / {lon_delta:.5f}°) : north-south {fixed_ns_m:7.0f} m | east-west {fixed_ew_m:7.0f} m")

# The east-west shortfall is the bug, and it is geometry, so it always reproduces.
assert naive_ew_m < radius_m * 0.85, (
    f"expected the un-scaled longitude delta to be far too narrow at latitude {user_lat}, "
    f"got {naive_ew_m:.0f} m against a {radius_m} m radius")
# The corrected box must reach past the radius on BOTH axes, or it is not a safe prefilter.
assert fixed_ns_m >= radius_m and fixed_ew_m >= radius_m, (
    f"the prefilter box must contain the circle, got ns={fixed_ns_m:.0f} m ew={fixed_ew_m:.0f} m")

# --- Probe A: false NEGATIVE. A point ~1,800 m due east — inside the radius, outside the box.
probe_a_lon = user_lon + 1800.0 / (M_PER_DEG_LON_EQ * cos_lat)
d_a = distance_to_user(user_lat, probe_a_lon)
in_naive_a = abs(probe_a_lon - user_lon) <= naive_delta
in_fixed_a = abs(probe_a_lon - user_lon) <= lon_delta
print(f"\n🔴 Probe A — a point due east: PostGIS says {d_a:.0f} m away")
print(f"   inside the 2 km radius? {d_a <= radius_m}")
print(f"   inside the NAIVE box?   {in_naive_a}   ← the user never sees this business")
print(f"   inside the FIXED box?   {in_fixed_a}")
assert d_a <= radius_m, f"probe A should be inside the radius, PostGIS says {d_a:.0f} m"
assert not in_naive_a, "probe A should have been MISSED by the naive box — the bug did not reproduce"
assert in_fixed_a, "probe A must be kept by the cos(latitude)-corrected box"

# --- Probe B: false POSITIVE. The corner of the naive box — outside the radius, inside the box.
d_b = distance_to_user(user_lat + naive_delta, user_lon + naive_delta)
print(f"\n🟠 Probe B — the naive box's NE corner: PostGIS says {d_b:.0f} m away")
print(f"   inside the 2 km radius? {d_b <= radius_m}   ← returned anyway, because it is in the box")
assert d_b > radius_m, f"the box corner should fall outside the radius, PostGIS says {d_b:.0f} m"

# --- Now the same three filters over the real seed data ---
def ids_in_box(dlat, dlon):
    cur.execute("""
        SELECT id FROM businesses
        WHERE latitude BETWEEN %s AND %s AND longitude BETWEEN %s AND %s;
    """, (user_lat - dlat, user_lat + dlat, user_lon - dlon, user_lon + dlon))
    return {row["id"] for row in cur.fetchall()}

cur.execute("""
    SELECT id FROM businesses
    WHERE ST_DWithin(location, ST_SetSRID(ST_MakePoint(%s, %s), 4326)::geography, %s);
""", (user_lon, user_lat, radius_m))
truth_ids = {row["id"] for row in cur.fetchall()}

naive_ids = ids_in_box(naive_delta, naive_delta)
fixed_ids = ids_in_box(lat_delta, lon_delta)

print(f"\n📊 Of the {len(truth_ids)} businesses genuinely within {radius_m} m:")
print(f"   naive box returned {len(naive_ids):3d}  → missed {len(truth_ids - naive_ids):3d}, "
      f"wrongly included {len(naive_ids - truth_ids):3d}")
print(f"   fixed box returned {len(fixed_ids):3d}  → missed {len(truth_ids - fixed_ids):3d}, "
      f"wrongly included {len(fixed_ids - truth_ids):3d} (corners — the second pass removes these)")

# The prefilter contract: the corrected box must be a SUPERSET of the true answer. False
# positives are fine (the second pass drops them); a false negative is a business nobody finds.
assert truth_ids <= fixed_ids, (
    f"the corrected box must contain every business inside the radius, "
    f"but it missed {sorted(truth_ids - fixed_ids)}")

conn.close()

### The Second Pass, Written Out

`ST_DWithin` already does prefilter-then-exact-distance for you. Here it is by hand, so you can
say in an interview what the second pass actually computes — and so we can check our own
arithmetic against the database's.

The Haversine formula treats the Earth as a sphere; `ST_Distance(geography)` uses the WGS-84
spheroid. They agree to within a few metres per kilometre, which is why we compare with a
tolerance rather than for equality.

In [ ]:
def haversine_m(lat1, lon1, lat2, lon2):
    """Great-circle distance in metres, on a sphere of mean Earth radius."""
    R = 6_371_008.8
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = p2 - p1
    dl = math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))

conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

# Pass 1 — index-friendly bounding box that CONTAINS the circle.
# Pass 2 — exact distance, in Python, on whatever survived.
cur.execute("""
    SELECT id, name, latitude, longitude,
           ST_Distance(location, ST_SetSRID(ST_MakePoint(%s, %s), 4326)::geography) AS postgis_m
    FROM businesses
    WHERE latitude BETWEEN %s AND %s AND longitude BETWEEN %s AND %s;
""", (user_lon, user_lat,
      user_lat - lat_delta, user_lat + lat_delta,
      user_lon - lon_delta, user_lon + lon_delta))
candidates = cur.fetchall()
conn.close()

haversine_by_id = {}
worst_rel_err = 0.0
for b in candidates:
    d = haversine_m(user_lat, user_lon, b["latitude"], b["longitude"])
    haversine_by_id[b["id"]] = d
    if b["postgis_m"] > 1.0:
        worst_rel_err = max(worst_rel_err, abs(d - b["postgis_m"]) / b["postgis_m"])

two_pass = {bid: d for bid, d in haversine_by_id.items() if d <= radius_m}

print(f"🔎 Pass 1 (bounding box) kept {len(candidates)} candidates out of 500")
print(f"🔎 Pass 2 (exact distance) kept {len(two_pass)}")
print(f"   PostGIS (ST_DWithin) says {len(truth_ids)}")
print(f"   worst sphere-vs-spheroid disagreement: {worst_rel_err * 100:.3f}%")

# Our Haversine must track PostGIS's spheroid distance closely. A big gap here means either
# the formula is wrong or `location` no longer matches the latitude/longitude columns.
assert worst_rel_err < 0.01, (
    f"Haversine disagrees with ST_Distance by {worst_rel_err * 100:.2f}% — "
    "check the formula, or check that `location` still matches the lat/lon columns")

# The two answers may differ only for businesses sitting within a metre or two of the boundary,
# where the sphere and the spheroid land on opposite sides of 2,000 m.
disagreements = set(two_pass) ^ truth_ids
for bid in sorted(disagreements):
    d = haversine_by_id[bid]
    assert abs(d - radius_m) < 25, (
        f"business {bid} is {d:.0f} m away — that is not a boundary rounding disagreement, "
        "the two-pass filter is genuinely wrong")
print(f"   boundary-only disagreements: {len(disagreements)}")

print("\n💡 This is exactly what ST_DWithin does under the hood, and what you should describe")
print("   in an interview: cheap index lookup first, exact distance on the survivors.")

## 🔍 Elasticsearch Geo-Distance Search

For production systems at Yelp's scale (10M businesses), **Elasticsearch** is the go-to choice. It natively supports:
- `geo_distance` queries (find points within a radius)
- `geo_bounding_box` queries (find points in a rectangle)
- Full-text search on business names
- Combined filters in a single query

Let's index our businesses into Elasticsearch and compare.

In [ ]:
# Step 1: Create an Elasticsearch index with a geo_point mapping

es = get_es()

INDEX_NAME = "businesses"

# Delete the index if it exists (for re-running this notebook)
if es.indices.exists(index=INDEX_NAME):
    es.indices.delete(index=INDEX_NAME)

# Create the index with proper mappings
es.indices.create(
    index=INDEX_NAME,
    body={
        "mappings": {
            "properties": {
                "name":        {"type": "text"},        # full-text searchable
                "description": {"type": "text"},
                "city":        {"type": "keyword"},     # exact match
                "category":    {"type": "keyword"},     # exact match
                "location":    {"type": "geo_point"},   # geospatial!
                "avg_rating":  {"type": "float"},
                "num_reviews": {"type": "integer"},
                "price_range": {"type": "integer"}
            }
        }
    }
)
print("✅ Created Elasticsearch index with geo_point mapping")

In [ ]:
# Step 2: Load businesses from Postgres into Elasticsearch

conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
cur.execute("""
    SELECT b.id, b.name, b.description, b.city, b.latitude, b.longitude,
           b.avg_rating, b.num_reviews, b.price_range, c.name AS category
    FROM businesses b
    JOIN categories c ON b.category_id = c.id
""")
businesses = cur.fetchall()
conn.close()

# Bulk index into Elasticsearch
from elasticsearch.helpers import bulk

actions = []
for biz in businesses:
    actions.append({
        "_index": INDEX_NAME,
        "_id": biz["id"],
        "_source": {
            "name": biz["name"],
            "description": biz["description"],
            "city": biz["city"],
            "category": biz["category"],
            "location": {"lat": float(biz["latitude"]), "lon": float(biz["longitude"])},
            "avg_rating": float(biz["avg_rating"] or 0),
            "num_reviews": biz["num_reviews"] or 0,
            "price_range": biz["price_range"]
        }
    })

success, errors = bulk(es, actions)
es.indices.refresh(index=INDEX_NAME)
print(f"✅ Indexed {success} businesses into Elasticsearch")
if errors:
    print(f"⚠️  {len(errors)} errors")

In [ ]:
# Step 3: Search for nearby businesses using Elasticsearch geo_distance

user_lat, user_lon = 40.758, -73.985  # Manhattan

start = time.time()
result = es.search(
    index=INDEX_NAME,
    body={
        "query": {
            "bool": {
                "filter": [
                    {
                        "geo_distance": {
                            "distance": "2km",
                            "location": {"lat": user_lat, "lon": user_lon}
                        }
                    }
                ]
            }
        },
        "sort": [
            {
                "_geo_distance": {
                    "location": {"lat": user_lat, "lon": user_lon},
                    "order": "asc",
                    "unit": "km"
                }
            }
        ],
        "size": 10
    }
)
es_time = (time.time() - start) * 1000

print(f"📍 Elasticsearch geo_distance search: {es_time:.2f} ms")
print(f"   Found {result['hits']['total']['value']} businesses within 2km\n")

for hit in result["hits"]["hits"]:
    src = hit["_source"]
    dist = hit["sort"][0]  # distance in km
    price = '$' * (src.get('price_range') or 1)
    print(f"   📍 {dist:.1f} km | ⭐ {src['avg_rating']:.1f} ({src['num_reviews']} reviews) | {price} | {src['name']}")

## 🔗 Combining Location + Category Filters

In the real Yelp, users search for **"coffee shops near me"** — combining location proximity with a category filter. Elasticsearch handles this with a `bool` query that stacks multiple conditions.

In [ ]:
# Combined search: restaurants within 5 km of San Francisco, sorted by rating

sf_lat, sf_lon = 37.7749, -122.4194  # San Francisco

result = es.search(
    index=INDEX_NAME,
    body={
        "query": {
            "bool": {
                "must": [
                    {"term": {"category": "Restaurants"}}  # exact category match
                ],
                "filter": [
                    {
                        "geo_distance": {
                            "distance": "5km",
                            "location": {"lat": sf_lat, "lon": sf_lon}
                        }
                    }
                ]
            }
        },
        "sort": [
            {"avg_rating": {"order": "desc"}},
            {
                "_geo_distance": {
                    "location": {"lat": sf_lat, "lon": sf_lon},
                    "order": "asc",
                    "unit": "km"
                }
            }
        ],
        "size": 10
    }
)

print(f"🍽️  Restaurants within 5 km of San Francisco (sorted by rating):")
print(f"   Total matches: {result['hits']['total']['value']}\n")

for hit in result["hits"]["hits"]:
    src = hit["_source"]
    dist = hit["sort"][1]  # geo_distance is second sort field
    price = '$' * (src.get('price_range') or 1)
    print(f"   ⭐ {src['avg_rating']:.1f} | 📍 {dist:.1f} km | {price} | {src['name']}")

## ⚡ Caching Geospatial Search Results in Redis

Search results for popular locations ("restaurants in Manhattan") are requested thousands of times per minute. We can cache these results in Redis to avoid hitting Elasticsearch repeatedly.

**Strategy**: Use a cache key based on the search parameters. Set a short TTL (e.g., 60 seconds) since business data changes slowly but search rankings may need to stay fresh.

In [ ]:
r = get_redis()

def search_nearby_cached(lat, lon, radius_km, category=None, limit=10):
    """
    Search for nearby businesses with Redis caching.
    Cache key encodes the search parameters so identical searches hit the cache.
    """
    # Build a cache key from the search parameters
    # Round lat/lon to 3 decimal places (~100m precision) to improve cache hit rate
    cache_key = f"search:{round(lat,3)}:{round(lon,3)}:{radius_km}:{category or 'all'}:{limit}"

    # Check cache first
    cached = r.get(cache_key)
    if cached:
        return json.loads(cached), True  # (results, was_cached)

    # Cache miss — query Elasticsearch
    query_body = {
        "query": {"bool": {"filter": [{"geo_distance": {"distance": f"{radius_km}km", "location": {"lat": lat, "lon": lon}}}]}},
        "sort": [{"avg_rating": {"order": "desc"}}, {"_geo_distance": {"location": {"lat": lat, "lon": lon}, "order": "asc", "unit": "km"}}],
        "size": limit
    }
    if category:
        query_body["query"]["bool"]["must"] = [{"term": {"category": category}}]

    result = es.search(index=INDEX_NAME, body=query_body)

    # Format results
    results = []
    for hit in result["hits"]["hits"]:
        src = hit["_source"]
        results.append({
            "name": src["name"], "category": src["category"],
            "avg_rating": src["avg_rating"], "num_reviews": src["num_reviews"],
            "distance_km": round(hit["sort"][1], 2)
        })

    # Store in cache with 60-second TTL
    r.setex(cache_key, 60, json.dumps(results))

    return results, False

# First call: cache MISS — hits Elasticsearch
start = time.time()
results, cached = search_nearby_cached(40.758, -73.985, 5, category="Restaurants")
t1 = (time.time() - start) * 1000
print(f"🔍 First search: {t1:.2f} ms (cached: {cached})")

# Second call: cache HIT — reads from Redis
start = time.time()
results, cached = search_nearby_cached(40.758, -73.985, 5, category="Restaurants")
t2 = (time.time() - start) * 1000
print(f"⚡ Second search: {t2:.2f} ms (cached: {cached})")
print(f"\n🚀 Speedup: {t1/t2:.1f}×\n")

for biz in results[:5]:
    print(f"   ⭐ {biz['avg_rating']:.1f} | 📍 {biz['distance_km']} km | {biz['name']}")

## 📊 Comparison: All Three Approaches

Let's measure them side by side.

In [ ]:
# Benchmark: 50 identical searches per approach.
#
# Fairness note: the Elasticsearch and Redis clients are long-lived, so we hold ONE Postgres
# connection open too. Opening a fresh connection inside the timed loop (which an earlier
# version of this cell did) costs a few milliseconds of TCP + auth and swamps the query
# itself — you end up benchmarking psycopg2.connect(), not the index.

def bench(label, fn, n=50):
    times = []
    for _ in range(n):
        start = time.time()
        fn()
        times.append((time.time() - start) * 1000)
    avg = sum(times) / len(times)
    print(f"  {label:<35} avg={avg:>7.2f} ms  min={min(times):>7.2f} ms  max={max(times):>7.2f} ms")
    return avg

print("⏱️  Benchmark: find businesses within 2 km of Manhattan (50 runs each)\n")

lat, lon = 40.758, -73.985

bench_conn = get_db()
bench_cur = bench_conn.cursor()

def naive_search():
    bench_cur.execute(
        "SELECT id, name FROM businesses WHERE latitude BETWEEN %s AND %s AND longitude BETWEEN %s AND %s LIMIT 10",
        (lat - 0.018, lat + 0.018, lon - 0.018, lon + 0.018))
    bench_cur.fetchall()

def postgis_search():
    bench_cur.execute(
        "SELECT id, name FROM businesses WHERE ST_DWithin(location, ST_SetSRID(ST_MakePoint(%s,%s),4326)::geography, 2000) LIMIT 10",
        (lon, lat))
    bench_cur.fetchall()

def es_search():
    es.search(index=INDEX_NAME, body={"query":{"bool":{"filter":[{"geo_distance":{"distance":"2km","location":{"lat":lat,"lon":lon}}}]}},"size":10})

# Warm the Redis cache first
r.delete(f"search:{round(lat,3)}:{round(lon,3)}:2:all:10")
search_nearby_cached(lat, lon, 2)  # populates cache

def redis_cached_search():
    search_nearby_cached(lat, lon, 2)

t_naive = bench("Naive bounding box (Postgres)", naive_search)
t_postgis = bench("PostGIS ST_DWithin (Postgres)", postgis_search)
t_es = bench("Elasticsearch geo_distance", es_search)
t_redis = bench("Redis cached search", redis_cached_search)

bench_conn.close()

# The only claim these numbers actually support at 500 rows is the caching one: a Redis GET
# does no query work at all, so it has to beat a round trip into Elasticsearch.
assert t_redis < t_es, f"expected the cached read to beat Elasticsearch, got redis={t_redis:.2f} ms es={t_es:.2f} ms"

print("\n⚠️  Be honest about what this measures. With 500 rows the naive sequential scan")
print(f"   ({t_naive:.2f} ms) may well beat the PostGIS index scan ({t_postgis:.2f} ms) — reading 500")
print("   rows straight through is cheaper than descending an R-tree. The spatial index wins")
print("   asymptotically, not on toy data; the query plans above are the real evidence.")
print("\n💡 Key insight: caching makes search effectively instant for repeated queries.")
print("   In a real system you'd use Elasticsearch for fresh queries and Redis for hot results.")

## 🧹 Cleanup

In [ ]:
# Clean up Elasticsearch index and Redis cache keys
r = get_redis()
keys = r.keys("search:*")
if keys:
    r.delete(*keys)
    print(f"🧹 Cleaned {len(keys)} Redis cache keys")

# We keep the ES index for the next notebooks
print("🧹 Cleanup complete (ES index kept for next notebooks)")

## 📚 Summary

### Key Takeaways

1. **Naive lat/lon queries are slow *and* wrong** — B-tree indexes can't efficiently handle 2D
   range queries, and a box built with equal deltas on both axes silently drops results east
   and west of you, because a degree of longitude shrinks with cos(latitude)
2. **PostGIS GIST indexes** use R-trees to make spatial queries fast — great for moderate scale
3. **Elasticsearch geo_distance** is purpose-built for search at scale — supports complex multi-filter queries
4. **Redis caching** makes repeated searches near-instant — use short TTLs to stay fresh
5. **Filter by distance first** — it's usually the most restrictive filter, shrinking the search space fastest
6. **A bounding box is a prefilter, never the answer** — size it so it *contains* the circle
   (divide the longitude delta by cos(latitude)), then run an exact distance pass on the survivors

### System Design Interview Tips

- Start with PostGIS if the interviewer discourages Elasticsearch
- Mention **geohashing** and **quadtrees** as alternative spatial indexing strategies
- Always discuss the **second-pass filter**: use the Haversine formula to calculate exact distance after the index narrows the search space
- If asked "why not just a bounding box?", give the two-part answer: the corners are false
  positives, and an un-scaled longitude delta gives false *negatives* — the failure users notice

### What This Toy Does NOT Do

- **No antimeridian or polar handling.** The `lon_delta = lat_delta / cos(lat)` correction we use
  blows up as cos(lat) → 0 near the poles, and a box straddling ±180° longitude has to be split
  in two. PostGIS and Elasticsearch handle both; our hand-rolled box does not.
- **No cell-boundary logic.** Geohash and quadtree schemes have the same problem in a different
  costume: a point 10 m away can land in the neighbouring cell, so you must always search the
  cell *and its neighbours*. We sidestep that here by letting the R-tree do the work.
- **A spherical Earth in the second pass.** Our Haversine helper assumes a sphere;
  `ST_Distance(geography)` uses the WGS-84 spheroid, so the two disagree by a few metres per
  kilometre. That is why the bounding box is deliberately oversized rather than exact.

### Next Up

In **Notebook 2**, we'll tackle **Review & Rating Aggregation** — how to efficiently calculate and update average ratings as reviews come in.